# 结果汇总与整理 Notebook

本 notebook 覆盖原本只通过脚本运行的结果整理流程：

- `scripts/summarize_results.py`：从 `summary.json` 生成 CSV/Markdown 总表
- `scripts/summarize_mape.py`：生成正式实验 MAPE 补充表
- `scripts/organize_results.py`：可选整理旧结果到按 horizon 分类的目录结构

这些单元只读取或整理已有结果，不会训练模型。

## 0. 配置

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for _ in range(6):
    if (ROOT / 'scripts').is_dir() and (ROOT / 'models').is_dir():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')

SUMMARY_CONFIG = {
    'datasets': 'ETTh1,ETTm1',
    'horizons': '24,48,96,168,336',
    'models': 'lstm,transformer,informer,autoformer,patchtst',
    'run_tags': 'formal_seed42',
    'output_prefix': 'formal_seed42_all',
}

DO_ORGANIZE_LEGACY_RESULTS = False
ORGANIZE_OVERWRITE = False
ORGANIZE_MODE = 'symlink'  # symlink / hardlink / copy

SUMMARY_CONFIG

## 1. 生成通用结果总表

In [ ]:
try:
    import pandas as pd
except ImportError as exc:
    raise ImportError('请先执行 pip install -r requirements.txt 安装 pandas。') from exc

from scripts.summarize_results import load_rows, parse_csv_list, parse_int_list, format_markdown_table

datasets = set(parse_csv_list(SUMMARY_CONFIG['datasets']))
horizons = set(parse_int_list(SUMMARY_CONFIG['horizons']))
models = set(parse_csv_list(SUMMARY_CONFIG['models']))
run_tags = set(parse_csv_list(SUMMARY_CONFIG['run_tags']))

rows = load_rows(ROOT / 'results', datasets, horizons, models, run_tags)
if not rows:
    raise FileNotFoundError('No matching *_summary.json files found')

df = pd.DataFrame(rows).sort_values(['dataset', 'horizon', 'model'])
csv_path = ROOT / 'results' / 'v1_csv' / 'formal' / f"{SUMMARY_CONFIG['output_prefix']}.csv"
md_path = ROOT / 'results' / 'v1_md' / 'formal' / f"{SUMMARY_CONFIG['output_prefix']}.md"
csv_path.parent.mkdir(parents=True, exist_ok=True)
md_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(csv_path, index=False)
md_path.write_text(format_markdown_table(df), encoding='utf-8')

print(f'Rows: {len(df)}')
print(f'CSV: {csv_path.relative_to(ROOT)}')
print(f'MD:  {md_path.relative_to(ROOT)}')
display(df.head(20))

## 2. 生成 MAPE 补充表

In [ ]:
from scripts.summarize_mape import (
    ROOT as MAPE_ROOT,
    CSV_DIR,
    MD_DIR,
    DETAIL_COLUMNS,
    AGG_COLUMNS,
    build_aggregate_rows,
    load_detail_rows,
    write_csv,
    write_markdown,
)

detail_rows = load_detail_rows()
aggregate_rows = build_aggregate_rows(detail_rows)
outputs = [
    (CSV_DIR / 'formal_seed42_mape.csv', detail_rows, DETAIL_COLUMNS),
    (MD_DIR / 'formal_seed42_mape.md', detail_rows, DETAIL_COLUMNS),
    (CSV_DIR / 'formal_seed42_mape_by_model.csv', aggregate_rows, AGG_COLUMNS),
    (MD_DIR / 'formal_seed42_mape_by_model.md', aggregate_rows, AGG_COLUMNS),
]
for path, table_rows, columns in outputs:
    if path.suffix == '.csv':
        write_csv(path, table_rows, columns)
    else:
        write_markdown(path, table_rows, columns)
    print(f'Saved {path.relative_to(ROOT)}')

print(f'Rows: detail={len(detail_rows)}, aggregate={len(aggregate_rows)}')
display(pd.DataFrame(detail_rows).head(20))
display(pd.DataFrame(aggregate_rows))

## 3. 可选：整理旧结果目录

默认不开启。只有需要把旧顶层结果链接/复制到 `results/h{horizon}/{dataset}/{model}/{run_tag}/` 结构时，再把 `DO_ORGANIZE_LEGACY_RESULTS` 改为 `True`。

In [ ]:
from scripts.organize_results import classify_result_pair, classify_summaries, write_index

if not DO_ORGANIZE_LEGACY_RESULTS:
    print('DO_ORGANIZE_LEGACY_RESULTS=False，跳过旧结果整理。')
else:
    summary_paths = sorted((ROOT / 'results').glob('*_summary.json'))
    linked_results = 0
    for summary_path in summary_paths:
        linked_results += classify_result_pair(summary_path, ORGANIZE_MODE, ORGANIZE_OVERWRITE)
    linked_summaries = classify_summaries(ORGANIZE_MODE, ORGANIZE_OVERWRITE)
    write_index(len(summary_paths), linked_summaries)
    print(f'Indexed experiment summaries: {len(summary_paths)}')
    print(f'Created/updated result view entries: {linked_results}')
    print(f'Created/updated aggregate summary entries: {linked_summaries}')